In [1]:
!pip install pydeseq2                                                     # PyDeSeq2 Installation


# Importing installed libraries and Python packages
try:
  import pandas as pd                                                     # For data manipulation
  import subprocess                                                       # For executing system commands from Python
  import os                                                               # For file management
  from tqdm import tqdm                                                   # For creating progress bars
  import numpy as np                                                      # For numerical computations
  import scipy.stats as stats                                             # For statistical analysis
  import numpy as np                                                      # For numerical computations
  import matplotlib.pyplot as plt                                         # For plotting
  import seaborn as sns                                                   # For creating advanced visualizations
  import pydeseq2                                                         # For differential gene expression analysis
  from sklearn.preprocessing import StandardScaler                        # For preprocessing the data
  from pydeseq2.dds import DeseqDataSet                                   # For Read counts modeling with the DeseqDataSet class
  from pydeseq2.ds import DeseqStats                                      # Statistical analysis with the DeseqStats class
  from pydeseq2.default_inference import DefaultInference
  from IPython.display import IFrame                                      # For Reading the web frame
  import warnings                                                         # For hiding the warning text
  from sklearn.exceptions import ConvergenceWarning
  warnings.filterwarnings("ignore", category=ConvergenceWarning)
  print("All Python packages imported successfully.")
except ImportError as e:
  print(f"Import failed: {e}")


# === Load counts matrix ===
counts_df = pd.read_csv("/content/merged_counts.txt", sep="\t", index_col=0)

# Remove featureCounts summary lines (those starting with "__")
counts_df = counts_df[~counts_df.index.str.startswith('__')]

# === Define metadata ===
df = pd.DataFrame({
    'SRR_ID': ["SRR087416", "SRR085471", "SRR085473", "SRR085474", "SRR085726", "SRR085725"],
    'Sample': [
        "Alzheimer's whole brain",
        "Normal brain, temporal lobe",
        "Alzheimer's brain, temporal lobe",
        "Normal brain, frontal lobe",
        "Alzheimer's brain, frontal lobe",
        "Normal whole brain"
    ]
})

# === Extract condition label (AD vs Normal) ===
metadata = df.copy()
metadata['condition'] = metadata['Sample'].apply(lambda x: 'AD' if "Alzheimer" in x else 'Normal')
metadata = metadata.set_index('SRR_ID')

# === Align metadata to counts matrix columns ===
metadata = metadata.loc[counts_df.columns]

                                       # === Run DESeq2 ===
# Define the inference object
inference = DefaultInference()

dds = DeseqDataSet(
    counts=counts_df.T,                # Transpose counts
    metadata=metadata,
    design_factors=["condition"],
    refit_cooks=True,
    inference=inference,
)
dds.deseq2()

# === Run Wald test ===
stat_res = DeseqStats(dds, contrast=("condition", "AD", "Normal"))
stat_res.summary()

# === Create output directory if not exists ===
os.makedirs("output", exist_ok=True)

# === Export results ===
res_df = stat_res.results_df.sort_values("padj")
res_df.to_csv("output/deseq2_results.csv")

# === Save normalized counts ===
#dds.normalized_counts.to_csv("output/normalized_counts.csv")
print("DESeq2 analysis complete. Results saved to 'output/deseq2_results.csv'")

# === Save normalized counts ===
#dds.normalized_counts.to_csv("output/normalized_counts.csv")
print("DESeq2 analysis complete. Results saved to 'PyDESeq_Output/deseq2_results.csv'")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.7/115.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.6/83.6 kB 7.0 MB/s eta 0:00:00
All Python packages imported successfully.
Using None as control genes, passed at DeseqDataSet initialization


<ipython-input-1-a1faf0fe45b9>:60: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(
Fitting size factors...
... done in 0.02 seconds.

Fitting dispersions...
... done in 28.83 seconds.

Fitting dispersion trend curve...
... done in 0.77 seconds.

Fitting MAP dispersions...
... done in 32.96 seconds.

Fitting LFCs...
... done in 21.12 seconds.

Calculating cook's distance...
... done in 0.03 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 12.08 seconds.



Log2 fold change & Wald test p-value: condition AD vs Normal
                  baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Geneid                                                                     
DDX11L1           0.000000             NaN       NaN       NaN       NaN   
WASH7P            0.122488       -0.203049  4.435055 -0.045783  0.963483   
MIR6859-1         0.000000             NaN       NaN       NaN       NaN   
MIR1302-2HG       0.000000             NaN       NaN       NaN       NaN   
MIR1302-2         0.000000             NaN       NaN       NaN       NaN   
...                    ...             ...       ...       ...       ...   
ND6           48540.187714        3.019565  0.978174  3.086940       NaN   
TRNE            156.678765        2.696438  0.727564  3.706120  0.000210   
CYTB         252602.633326        2.033413  0.602297  3.376095  0.000735   
TRNT            141.565601        1.951247  0.560513  3.481183  0.000499   
TRNP            792.335924 